In [ ]:
# import gc, os
# import json
# import time
# from pathlib import Path
# from typing import List, Tuple, Dict

# import pyarrow
# import matplotlib.pyplot as plt
# import polars as pl
# import pandas as pd
# import numpy  as np
# import optuna
# from sklearn.model_selection import train_test_split, StratifiedKFold
# from catboost import CatBoostClassifier, Pool



# def init_data_start(target_id: str, train, target):
#     # Этап 1: загрузка данных
#     target_by_id = target[target_id].to_numpy()

#     # Этап 2: разделяем на индексы на valid/train
#     n_samples = len(train)
#     train_idx, valid_idx = train_test_split(
#         np.arange(n_samples),
#         test_size=0.2,
#         random_state=42,
#         stratify=target_by_id
#     )

#     # Этап 3: возьмем по деланным индексам данные
#     X_train, y_train = train[train_idx], target_by_id[train_idx]
#     X_valid, y_valid = train[valid_idx], target_by_id[valid_idx]

#     # Этап 4: выбираем cat_feature с ними работает catboost
#     cat_feature_names = [col for col in train.columns if col.startswith("cat_feature")]

#     # Этап 5: для catboost pool лучше
#     train_pool = Pool(X_train, y_train, cat_feature_names)
#     valid_pool = Pool(X_valid, y_valid, cat_feature_names)

#     return train_pool, valid_pool, X_train, y_train


# def objective(trial: optuna.Trial, train: Pool, valid: Pool):
#     params = {
#         'eval_metric': 'AUC',
#         'verbose': 0, # отчёт каждые N новых деревьев
#         'use_best_model': True,
#         'early_stopping_rounds': 50, # если метрика не растёт 50 деревьев, пруним
#         'task_type': 'GPU',
#         'thread_count': 20,  # кол-во используемых потоков(-1 макс)
#         'iterations': 6_000, # специально много, потому что ставка не на это
#         'depth': trial.suggest_int('depth', 4, 7),
#         'learning_rate': trial.suggest_float('learning_rate', .01, .3, log=True),
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1., 50., log=True),
#         'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
#         'random_strength': trial.suggest_int('random_strength', 1, 7),
#     }

#     model = CatBoostClassifier(**params)
#     model.fit(train, eval_set=valid)

#     best_score = model.get_best_score()['validation']['AUC']

#     # ДЛЯ ПРУНИНГА
#     scores = model.get_evals_result()['validation']['AUC']
#     for epoch, score in enumerate(scores, 1):
#         trial.report(score, epoch)

#         if trial.should_prune():
#             raise optuna.TrialPruned()
    
#     # ДЛЯ OPTUNA
#     return best_score


# def get_bst_par(train: Pool, valid: Pool):
#     study = optuna.create_study(
#         direction='maximize',
#         sampler=optuna.samplers.TPESampler(seed=42),
#         pruner=optuna.pruners.MedianPruner(
#             n_startup_trials=1, # кол-во честных попыток (без прунинга)
#             n_warmup_steps=20
#         )
#     )
#     study.optimize(
#         lambda trial: objective(trial, train, valid),
#         n_trials=2,    # ограничение в 30попыток поиска параметров
#         gc_after_trial=True,
#     )
    
#     return study.best_params


# def get_bst_features(
#         best_params,
#         train: Pool,
#         valid: Pool,
#         min_features: int = 50,     # мин. кол-во признаков
#         max_features: int = 170,    # макс. кол-во признаков
#         coverage: float = 0.95      # оставляем признаки, дающие 95% общей важности
#     ):
#     model = CatBoostClassifier(
#         **best_params,
#         task_type='GPU',
#         verbose=False
#     )
#     model.fit(train, eval_set=valid)

#     # Feature Id    Importances
#     feature_importance = model.get_feature_importance(prettified=True)

#     feature_importance['cumsum'] = feature_importance['Importances'].cumsum()

#     threshold = feature_importance['Importances'].sum() * coverage
#     matrix = feature_importance[feature_importance['cumsum'] >= threshold]

#     n_features = matrix.index[0] + 1 if len(matrix) > 0 else len(matrix)

#     N = max(min_features, min(n_features, max_features))
#     best_features = feature_importance['Feature Id'].head(N).tolist()

#     print(f'Выбрано {N} признаков')

#     return best_features


# def oof_one_target(best_features, best_params, X_train, y_train):

#     X_train = X_train.select(best_features)
#     category = [col for col in X_train.columns if col.startswith('cat_feature')]

#     kf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
#     preds_model = np.zeros(len(X_train))

#     for train_idx, valid_idx in kf.split(X_train, y_train):

#         X_tr, y_tr   = X_train[train_idx], y_train[train_idx]
#         X_val, y_val = X_train[valid_idx], y_train[valid_idx]

#         train_fold = Pool(data=X_tr,  label=y_tr,  cat_features=category)
#         valid_fold = Pool(data=X_val, label=y_val, cat_features=category)

#         model = CatBoostClassifier(
#             **best_params,
#             task_type='GPU',
#             verbose=False
#         )
#         model.fit(train_fold, eval_set=valid_fold)
#         preds_model[valid_idx] = model.predict_proba(valid_fold)[:,1]

#         del model, train_fold, valid_fold, X_tr, X_val
#         gc.collect()

#     return preds_model, category 


# def save_snapshot(best_params, best_features, target_idx, X_train, y_train, category, oof_score):
#     X_train = X_train[best_features]
#     model = CatBoostClassifier(
#         **best_params,
#         task_type='GPU',
#         verbose=False
#     )
    
#     X_all_train = Pool(data=X_train, label=y_train, cat_features=category)
#     model.fit(X_all_train)

#     Path('snapshots').mkdir(exist_ok=True)
#     Path('configs').mkdir(exist_ok=True)

#     model.save_model(f'snapshots/{target_idx}.cbm')

#     data = {
#         'target': target_idx,
#         'best_score': oof_score,
#         'best_params': best_params,
#         'best_features': best_features,
#         'category': category,
#     }
    
#     with open(f'configs/{target_idx}.json', 'w', encoding='utf-8') as file:
#         json.dump(data, file, indent=4)


# def save_oof_column(target_idx: str, preds: np.ndarray, filepath: str = 'meta_data.parquet'):
#     if os.path.exists(filepath):
#         meta_df = pl.read_parquet(filepath)
#         meta_df = meta_df.with_columns(pl.Series(target_idx, preds))
#     else:
#         meta_df = pl.DataFrame({target_idx: preds})
    
#     meta_df.write_parquet(filepath)

#     print(f"Cтолбец '{target_idx}' сохранён в {filepath}")

/home/d.golomolzin/kiber-polka/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# train_path = 'data/train_main_features_typed.parquet'
# target_path = 'data/train/train_target.parquet'

# train = pl.read_parquet(train_path).drop('customer_id')
# target = pl.read_parquet(target_path).drop('customer_id')
# columns_tar = target.columns

In [ ]:
# time_all = time.time()
# target_idx = columns_tar[0]
# print(f"TARGET: {target_idx}")
# time_target = time.time()

TARGET: target_1_1


In [ ]:
# print('Этап 1: получаем data')
# time_stage = time.time()
# train_pool, valid_pool, X_train, y_train = init_data_start(target_idx, train, target)
# print(f'Этап 1 выполнен за {(time.time() - time_stage) / 60:.2f} мин.')

Этап 1: получаем data
Этап 1 выполнен за 0.01 мин.


In [ ]:
# print('Этап 2: получаем лучшие гиперпараметры таргета')
# time_stage = time.time()
# best_params = get_bst_par(train_pool, valid_pool)
# print(f'Этап 2 выполнен за {(time.time() - time_stage) / 60:.2f} мин.')

[I 2026-03-10 03:54:26,939] A new study created in memory with name: no-name-39683260-3556-4c2b-92ec-a319a04b0bf7


Этап 2: получаем лучшие гиперпараметры таргета


Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-03-10 03:54:57,673] Trial 0 finished with value: 0.8791371583938599 and parameters: {'depth': 5, 'learning_rate': 0.2536999076681772, 'l2_leaf_reg': 17.524101118128137, 'min_data_in_leaf': 30, 'random_strength': 2}. Best is trial 0 with value: 0.8791371583938599.
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-03-10 03:55:03,788] Trial 1 pruned. 


Этап 2 выполнен за 0.62 мин.


In [ ]:
# print('Этап 3: получаем лучшие фичи в датасете для таргета')
# time_stage = time.time()
# best_features = get_bst_features(best_params, train_pool, valid_pool)
# print(f'Этап 3 выполнен за {(time.time() - time_stage) / 60:.2f} мин.')

Этап 3: получаем лучшие фичи в датасете для таргета
Выбрано 61 признаков
Этап 3 выполнен за 0.86 мин.


In [ ]:
# print('Этап 4: получаем столбец предсказаний по target_id')
# time_stage = time.time()
# preds_target, category = oof_one_target(best_features, best_params, X_train, y_train)
# print(f'Этап 4 выполнен за {(time.time() - time_stage) / 60:.2f} мин.')

Этап 4: получаем столбец предсказаний по target_id
Этап 4 выполнен за 0.78 мин.


In [ ]:
# print('Этап 5: сохраняем данные о модели')
# time_stage = time.time()
# from sklearn.metrics import roc_auc_score
# oof_auc = roc_auc_score(y_train, preds_target)
# save_snapshot(best_params, best_features, target_idx, X_train, y_train, category, oof_auc)
# print(f'Этап 5 выполнен за {(time.time() - time_stage) / 60:.2f} мин.')

Этап 5: сохраняем данные о модели
Этап 5 выполнен за 0.45 мин.


In [ ]:
# print('Этап 6: сохраняем preds_target для каждого таргета')
# time_stage = time.time()
# save_oof_column(target_idx, preds_target, 'meta_data.parquet')
# print(f'Этап 6 выполнен за {(time.time() - time_stage) / 60:.2f} мин.')

# gc.collect()

Этап 6: сохраняем preds_target для каждого таргета
Cтолбец 'target_1_1' сохранён в meta_data.parquet
Этап 6 выполнен за 0.00 мин.


33

In [8]:
import polars as pl

df = pl.read_parquet(r'/home/d.golomolzin/kiber-polka/meta_data.parquet')
df = df.drop('target_2_6')
df.write_parquet('meta_data.parquet')

In [ ]:
import polars as pl

m = pl.read_parquet(r'/home/d.golomolzin/kiber-polka/meta_data.parquet')
m
# чет выходы решили использовать float64 по полной..... особенно target_2_8

target_1_1,target_1_2,target_1_3,target_1_4,target_1_5,target_2_1,target_2_2,target_2_3,target_2_4,target_2_5,target_2_6,target_2_7,target_2_8,target_3_1,target_3_2,target_3_3,target_3_4,target_3_5,target_4_1,target_5_1
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.012091,0.000921,0.002333,0.129122,0.001371,0.000485,0.011462,0.000106,0.00821,0.000922,0.003061,0.000031,0.000003,0.087494,0.00811,0.000472,0.00004,0.003872,0.009092,0.005858
0.000267,0.001879,0.082059,0.0049,0.000119,0.021912,0.004839,0.000245,0.002721,0.000418,0.004062,0.000028,0.000008,0.079334,0.007919,0.000481,0.000077,0.000292,0.003657,0.012052
0.000205,0.002023,0.012272,0.021234,0.000052,0.00526,0.002931,0.000535,0.002832,0.004882,0.003292,0.000045,0.000008,0.022032,0.148681,0.002634,0.00718,0.000223,0.002466,0.002036
0.056534,0.000826,0.002374,0.137039,0.00073,0.002602,0.025768,0.000229,0.004186,0.000327,0.007853,0.000089,0.000128,0.042651,0.00901,0.000856,0.000998,0.020914,0.015828,0.003513
0.003375,0.000671,0.004306,0.02001,0.001106,0.001342,0.005764,0.001517,0.005383,0.000759,0.003518,0.000159,7.6421e-7,0.116552,0.011192,0.000343,0.000043,0.016977,0.003475,0.028835
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.001771,0.006072,0.068447,0.002439,0.000662,0.001585,0.00565,0.000512,0.008993,0.000521,0.003133,0.000227,0.000002,0.158121,0.006533,0.000363,0.000018,0.04395,0.008277,0.002676
0.012543,0.000198,0.003661,0.001819,0.004271,0.003116,0.15584,0.001763,0.003833,0.002918,0.002981,0.000306,0.000001,0.311631,0.011171,0.003773,0.000057,0.00012,0.002197,0.017809
0.001783,0.005315,0.183688,0.014341,0.000422,0.000735,0.003653,0.002972,0.005594,0.000087,0.002439,0.000003,0.000001,0.128722,0.042317,0.000507,0.000723,0.000015,0.002809,0.012081
